# Gated Recurrent Unit: Entrenamiento y Evaluación

In [1]:
import os
import random
import sys
import warnings
from pathlib import Path
import optuna
import time

import numpy as np
import pandas as pd
import matplotlib.dates as mdates
import torch
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.models.gru_wrapper import GRUModel
from src.eval.scorer import VolatilityScorer
from src.trainer.walk_forward import WalkForwardTrainer
from src.data.dataset import TimeSeriesDataset
from src.utils import trainer_toolkit as tt
from src.utils import ts_toolkit as tst
from src.utils import test_toolkit as ts

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

OUT_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results" / "gru"
MODELS_PATH = PROJECT_ROOT / "models"

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(OUT_PATH / "brent_train.parquet")
y_train = df_train["log_returns"].values.astype("float64")
rs_train = df_train["rs_daily"].values.astype("float64")

df_test = pd.read_parquet(OUT_PATH / "brent_test.parquet")
y_test = df_test["log_returns"].values.astype("float64")
rs_test = df_test["rs_daily"].values.astype("float64")

assert not np.isnan(y_train).any() and not np.isnan(rs_train).any(), "NaNs en train"
assert not np.isnan(y_test).any() and not np.isnan(rs_test).any(), "NaNs en test"
assert df_train.index.max() < df_test.index.min(), "Solapamiento temporal train/test"

print("=" * 70)
print(f"Train: {len(y_train)} obs  ({df_train.index[0].date()} -> {df_train.index[-1].date()})")
print(f"Test:  {len(y_test)} obs  ({df_test.index[0].date()} -> {df_test.index[-1].date()})")
print("=" * 70)

Train: 2171 obs  (2014-01-03 -> 2022-11-03)
Test:  953 obs  (2022-11-07 -> 2026-08-21)


:::{admonition} Resumen
:class: info
- La búsqueda de hiperparámetros con Optuna (trial 26, con QLIKE de validación de 0.429903 y detención en el trial 42 de 50) seleccionó una arquitectura GRU con ventana de 7 días, 128 unidades ocultas y 1 capa recurrente.
- En entrenamiento, el modelo convergió sin señales de sobreajuste, sin autocorrelación ni efectos ARCH en los rezagos evaluados y con residuos de colas pesadas mejor descritos por una distribución $t$ de Student que por la normal.
- En evaluación fuera de muestra (*walk-forward*, 113.16 minutos para 953 observaciones), el QLIKE aumentó por encima del umbral de tolerancia de 0.05 frente a entrenamiento, acompañado de saltos en HMSE y HMAE y persistencia de autocorrelación residual en los rezagos 20 y 30, sin efectos ARCH ni dependencia no lineal en dimensión 2.
- En el episodio de volatilidad de 2026, el modelo siguió la dinámica general con oscilaciones marcadas, pero subestimó la magnitud del pico principal de marzo y mostró un cuantil inferior extremo atípico en la cola izquierda.
:::

## Introducción

La Unidad Recurrente con Compuertas (*Gated Recurrent Unit*, GRU) es una arquitectura de red neuronal recurrente diseñada para modelar dependencias temporales en datos secuenciales. A diferencia de las redes neuronales tradicionales, incorpora mecanismos de memoria que permiten conservar y actualizar información de observaciones anteriores mediante compuertas entrenables.

En el contexto de las series financieras, esta arquitectura ofrece una alternativa a los modelos econométricos tradicionales, al permitir que la relación entre los retornos históricos y la volatilidad futura sea aprendida directamente a partir de los datos. Su estructura recurrente facilita la identificación de patrones temporales y cambios en la dinámica de los retornos, sin establecer previamente una ecuación paramétrica para la evolución de la varianza condicional.

El objetivo de esta sección es presentar, entrenar y evaluar un modelo GRU para la predicción de la volatilidad del crudo Brent bajo el protocolo del benchmark. Se estudia el efecto de distintas longitudes de secuencia sobre el desempeño predictivo de la red, manteniendo las condiciones de información y evaluación establecidas para los demás modelos del proyecto.

### Arquitectura

Una red GRU procesa secuencialmente las observaciones de entrada y actualiza un estado oculto que resume la información temporal disponible. Para controlar este proceso, utiliza dos mecanismos principales: la compuerta de actualización (*update gate*) y la compuerta de reinicio (*reset gate*).

<div align="center">
  <img src="../diagrams/gru_cell_diagram.jpeg" alt="Diagrama de Celda GRU" width="80%">
</div>

La compuerta de actualización determina qué proporción de la información anterior debe conservarse, mientras que la compuerta de reinicio regula la influencia del estado previo en la construcción de nueva información. Para una observación de entrada $\mathbf{x}_t$, estas operaciones se expresan como:

$$
\mathbf{z}_t = \sigma\left(\mathbf{W}_z\mathbf{x}_t+\mathbf{U}_z\mathbf{h}_{t-1}+\mathbf{b}_z\right),
$$

$$
\mathbf{r}_t = \sigma\left(\mathbf{W}_r\mathbf{x}_t+\mathbf{U}_r\mathbf{h}_{t-1}+\mathbf{b}_r\right),
$$

donde $\mathbf{z}_t$ representa la compuerta de actualización, $\mathbf{r}_t$ la compuerta de reinicio, $\sigma$ la función sigmoide y $\mathbf{h}_{t-1}$ el estado oculto anterior.

El estado candidato se calcula mediante:

$$
\tilde{\mathbf{h}}_t =
\tanh\left(
\mathbf{W}_h\mathbf{x}_t+
\mathbf{U}_h(\mathbf{r}_t\odot\mathbf{h}_{t-1})+
\mathbf{b}_h
\right),
$$

y el nuevo estado oculto se obtiene como:

$$
\mathbf{h}_t =
(1-\mathbf{z}_t)\odot\mathbf{h}_{t-1}
+
\mathbf{z}_t\odot\tilde{\mathbf{h}}_t,
$$

donde $\odot$ representa el producto de Hadamard (elemento a elemento). 

Una vez calculado el estado oculto final de la secuencia, $\mathbf{h}_t \in \mathbb{R}^{H}$ (donde $H$ es el número de unidades ocultas), este vector resume la información de la ventana temporal. Para generar un pronóstico escalar de la varianza condicional, este vector se proyecta a través de una capa densa (*fully connected layer*) con pesos $\mathbf{w} \in \mathbb{R}^{H}$ y sesgo $b \in \mathbb{R}$:

$$
y_t = \mathbf{w}^{\top}\mathbf{h}_t + b.
$$

Para garantizar que la varianza pronosticada sea estrictamente positiva (un requisito matemático para la volatilidad), el resultado escalar $y_t$ pasa por una función de activación *Softplus*, la cual es una versión suavizada de la función ReLU:

$$
\hat{\sigma}_t^2 = \text{Softplus}(y_t) = \ln(1 + e^{y_t}),
$$

donde $\boldsymbol{\theta}$ agrupa los pesos y sesgos tanto de las unidades GRU como de la capa de salida.

<div align="center">
  <img src="../diagrams/gru_arquitecture.jpeg" alt="Diagrama de Arquitectura GRU" width="80%">
</div>

Estos mecanismos permiten que la red aprenda qué información temporal conservar, modificar o descartar. A diferencia del perceptrón multicapa (*Multilayer Perceptron*, MLP), que recibe los rezagos como un vector de características, la GRU procesa las observaciones como una secuencia ordenada, incorporando explícitamente la dependencia temporal en su arquitectura.

Para aplicar esta arquitectura al pronóstico de la volatilidad del crudo Brent, se partió del retorno logarítmico diario $r_t$. Dado que los retornos financieros presentan magnitudes pequeñas (del orden de $10^{-3}$), su ingreso directo a la red provoca el desvanecimiento de los gradientes durante la retropropagación, impidiendo el aprendizaje. Por esta razón, los retornos fueron previamente estandarizados mediante una transformación Z-score (media cero y varianza unitaria). Para evitar cualquier fuga de información (*data leakage*), el escalador (`StandardScaler`) se ajustó exclusivamente con el primer 80 % de los datos de cada fold de entrenamiento, aplicando únicamente la transformación sobre el 20 % restante de validación y sobre el conjunto de prueba. 

:::{admonition} Información: Transformación Z-score
:class: info

La estandarización Z-score es una técnica de preprocesamiento que transforma una variable para que tenga una media de cero y una desviación estándar de uno. Dado un retorno logarítmico $r_t$, su versión estandarizada $\tilde{r}_t$ se calcula mediante:

$$
\tilde{r}_t = \frac{r_t - \mu_{\text{train}}}{\sigma_{\text{train}}}
$$

donde $\mu_{\text{train}}$ y $\sigma_{\text{train}}$ son la media y la desviación estándar calculadas sobre el conjunto de entrenamiento. 
:::

Sea $\tilde{r}_t$ el retorno estandarizado, para una ventana temporal de longitud $L$, la secuencia de entrada se define como:

$$
\mathbf{Z}_t =
\begin{bmatrix}
\tilde{r}_{t-L}\\
\tilde{r}_{t-L+1}\\
\vdots\\
\tilde{r}_{t-1}
\end{bmatrix}
\in \mathbb{R}^{L\times 1}.
$$

El modelo completo (la GRU y su capa de salida) genera el pronóstico de la varianza condicional del día inmediatamente posterior mediante la función $f_{\boldsymbol{\theta}}(\mathbf{Z}_t)$.

El problema se aborda mediante una estrategia de predicción de un paso hacia adelante (*one-step-ahead forecasting*). El entrenamiento consiste en encontrar los parámetros $\boldsymbol{\theta}$ que minimicen la pérdida QLIKE sobre las observaciones disponibles, dado que esta métrica penaliza asimétricamente las subestimaciones de la volatilidad. El problema de optimización se expresa como:

$$
\boldsymbol{\theta}^{*}
=\arg\min_{\boldsymbol{\theta}}
\frac{1}{N}
\sum_{i=1}^{N}
\mathcal{L}_{\text{QLIKE}}
\left(
RS_{i},
\hat{RS}_{i}
\right),
$$

donde $N$ es el número de secuencias de entrenamiento y $RS_i$ es la varianza de Rogers-Satchell utilizada como *target*.

Los parámetros se actualizan mediante retropropagación a través del tiempo (*backpropagation through time*, BPTT) y el optimizador Adaptive Moment Estimation (Adam). La elección del número de unidades recurrentes, las capas ocultas, la longitud de las secuencias ($L$) y los parámetros de entrenamiento (tasa de aprendizaje, *dropout*) influye directamente sobre la capacidad de ajuste y generalización del modelo.

La implementación de la GRU se realiza utilizando la librería `PyTorch` en su versión `2.13.0`, aprovechando su módulo de diferenciación automática para el cálculo de gradientes y la optimización de los pesos.

## Entrenamiento

Para la selección de hiperparámetros del modelo GRU, se mantuvo el mismo protocolo experimental (Walk-Forward) y el mismo presupuesto computacional que en los modelos paramétricos. A diferencia del GARCH y el EGARCH, la búsqueda de Optuna exploró la arquitectura de la red neuronal y sus parámetros de regularización.

Para adaptar la serie temporal al formato requerido por la red, se construyó una matriz de rezagos (*lag matrix*) para cada ventana del esquema Walk-Forward. Siguiendo los principios de validación cruzada para series temporales (`timeseriescv`), cada fila de esta matriz contenía una secuencia histórica de longitud `look_back`, la cual se utilizó como vector de entrada para predecir el paso inmediatamente posterior. Este enfoque garantizó la integridad temporal de los datos y evitó cualquier tipo de fuga de información futura (*look-ahead bias*) durante el entrenamiento.

El espacio de búsqueda incluyó la longitud de la ventana de entrada (`look_back`), el número de neuronas en la capa oculta (`hidden_size`), la cantidad de capas recurrentes, la tasa de abandono (`dropout`) y la tasa de aprendizaje. Para mitigar el sobreajuste, se incorporó también la penalización $\ell_2$ (`weight_decay`). El entrenamiento se configuró con un máximo de 200 épocas y una paciencia de 20 para la parada temprana interna de la red. El desempeño se siguió evaluando mediante la pérdida QLIKE sobre los conjuntos de validación.

In [ ]:
def gru_search_space(trial: optuna.Trial) -> dict:
    return {
        "look_back": trial.suggest_categorical("look_back", [4, 7, 14, 21, 28]),
        "hidden_size": trial.suggest_categorical("hidden_size", [32, 64, 128]),
        "num_layers": trial.suggest_int("num_layers", 1, 2),
        "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "max_epochs": 200,
        "patience": 20,
        "retrain_freq": 1,
        "seed": 42
    }

exec_id = 2
current_warmup = 1200 + (exec_id * 10)

exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

trainer = WalkForwardTrainer(
    model_class=GRUModel,
    scorer=VolatilityScorer(),
    output_dir=exec_dir,
    n_splits=5, warmup=current_warmup,
    embargo=1, n_trials=50,
    suggest_params_fn=gru_search_space,
    early_stop_rounds=15
)

trainer.run(y_train, rs_train)

Respecto a la eficiencia del algoritmo de búsqueda, este se detuvo en el trial 42 sin alcanzar el límite máximo de 50 evaluaciones, debido a la activación del mecanismo de parada temprana (*Early Stopping*) tras 15 intentos consecutivos sin mejoras. El mecanismo de poda (*Pruning*) descartó 8 configuraciones, lo que evidenció la utilidad del *Median Pruner* para abortar tempranamente arquitecturas con bajo desempeño predictivo durante las primeras particiones, optimizando así el presupuesto computacional. La mejor configuración registró un QLIKE de 0.4299.

El ajuste de los pesos bajo el esquema Walk-Forward tuvo una duración total de 9.16 minutos (549.40 segundos), con un tiempo medio de 2.95 segundos por pliegue. Es fundamental precisar que este costo corresponde exclusivamente al entrenamiento de la red. La evaluación completa del modelo, que incluyó el reentrenamiento iterativo durante la generación de pronósticos (*forecast*) para las observaciones fuera de muestra, elevó el tiempo de entrenamiento total a aproximadamente 14.59 horas. Este alto costo computacional frente a los 9.12 y 12.54 segundos de los modelos GARCH y EGARCH reflejó el compromiso necesario para dotar al modelo de mayor flexibilidad no paramétrica.

In [2]:
resumen_data = []

exec_id = 2
exec_dir = RESULTS_PATH / f"exec_{exec_id}"

tiempo_total_rep = 0.0
tiempo_promedio_fold = 0.0
num_folds = 0
total_trials, completos, podados, fallidos = 0, 0, 0, 0
mejor_qlike = "-"

csv_files = list(exec_dir.glob("*_results.csv"))
if csv_files:
    df_csv = pd.read_csv(csv_files[0])
    tiempo_total_rep = df_csv["train_time_sec"].sum()
    tiempo_promedio_fold = df_csv["train_time_sec"].mean()
    num_folds = len(df_csv)
else:
    print(f"No se encontró el CSV en {exec_dir}")

db_path = exec_dir / "optuna_study.db"
if db_path.exists():
    try:
        storage_url = f"sqlite:///{db_path.resolve()}"
        study = optuna.load_study(study_name="GRUModel_study", storage=storage_url)

        total_trials = len(study.trials)
        completos = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
        podados  = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
        fallidos = len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])

        mejor_qlike = round(study.best_value, 4)

    except Exception as e:
        print(f"Error leyendo BD en {exec_dir}: {e}")
else:
    print(f"No se encontró la BD en {db_path}")

resumen_data.append({
    "Total Trials": total_trials,
    "Completos": completos,
    "Podados": podados,
    "Mejor QLIKE": mejor_qlike,
    "Tiempo Total (s)": round(tiempo_total_rep, 2),
    "Tiempo por Fold (s)": round(tiempo_promedio_fold, 3)
})

df_search_params = pd.DataFrame(resumen_data)
tt.display_styled_df(df=df_search_params, decimals=4, align_left_first=False)

Total Trials,Completos,Podados,Mejor QLIKE,Tiempo Total (s),Tiempo por Fold (s)
42,34,8,0.4299,549.4000,2.9540


El ranking de configuraciones mostró que las cinco mejores opciones presentaron una pérdida QLIKE en el rango de 0.430 a 0.447. Las diferencias entre estas configuraciones fueron menores que la variabilidad entre folds ($\sigma \approx 0.11$), lo que indicó un empate estadístico en su desempeño predictivo. A pesar de que los hiperparámetros continuos (tasa de aprendizaje, *weight decay* y *dropout*) variaron sin un patrón claro, se observó un consenso en la arquitectura base: todas seleccionaron una ventana de memoria (*lookback*) de 7 días, una sola capa recurrente y, en su mayoría, 128 neuronas ocultas.

Esta convergencia arquitectónica aportó información relevante sobre la dinámica de los datos. Aunque el espacio de búsqueda exploró ventanas temporales más largas (14, 21 y 28 días), la red convergió de forma consistente hacia los 7 días. Esto sugirió que la GRU priorizó una memoria de corto plazo (aproximadamente una semana bursátil) para capturar el agrupamiento de volatilidad inmediato, evitando incorporar ruido de periodos más antiguos. De igual forma, la elección de 128 neuronas en la mayoría de las mejores configuraciones junto con una sola capa evidenció que la red logró un equilibrio al aumentar su capacidad de representación en el espacio latente, pero manteniendo una sola capa recurrente para mitigar el riesgo de sobreajuste.

In [3]:
exec_dir = RESULTS_PATH / "exec_2"

df_gru = pd.read_csv(exec_dir / "GRUModel_results.csv")

df_gru["lr"] = df_gru["learning_rate"].apply(lambda x: f"{x:.4f}")
df_gru["wd"] = df_gru["weight_decay"].apply(lambda x: f"{x:.1e}")

cols_agrupar = ["look_back", "hidden_size", "num_layers", "dropout", "lr", "wd"]
df_agg = df_gru.groupby(cols_agrupar)["qlike"].agg(["mean", "std"]).reset_index()
df_agg["QLIKE"] = df_agg.apply(lambda x: f"{x['mean']:.3f} ± {x['std']:.3f}", axis=1)
df_top5 = df_agg.sort_values(by="mean").head(5)

df_top5 = df_top5.rename(columns={
    "look_back": "Lookback",
    "hidden_size": "Neuronas",
    "num_layers": "Capas",
    "dropout": "Dropout",
    "lr": "LR",
    "wd": "Weight Decay"
})

df_final = df_top5[["Lookback", "Neuronas", "Capas", "Dropout", "LR", "Weight Decay", "QLIKE"]]

tt.display_styled_df(df=df_final, align_left_first=False, decimals=3)

Lookback,Neuronas,Capas,Dropout,LR,Weight Decay,QLIKE
7,128,1,0.027,0.0023,7.7e-04,0.430 ± 0.109
7,128,1,0.045,0.0023,7.7e-04,0.438 ± 0.109
7,128,1,0.013,0.0010,8.1e-04,0.440 ± 0.114
7,32,1,0.084,0.0018,6.8e-04,0.442 ± 0.090
7,128,1,0.043,0.0010,7.0e-04,0.447 ± 0.116


Las dos configuraciones con menor pérdida compartieron exactamente la misma arquitectura base (Lookback de 7 días, 128 neuronas y 1 capa) y el mismo *weight decay* ($7.7 \times 10^{-4}$), difiriendo en su tasa de *dropout* (0.027 frente a 0.045) y registrando valores medios de QLIKE de 0.430 y 0.438, respectivamente. Dado que esta diferencia es pequeña frente a la variabilidad observada entre los folds ($\sigma \approx 0.109$), ambas presentan un desempeño predictivo similar en términos de media y dispersión entre particiones.

Un hallazgo relevante fue la magnitud de la desviación estándar del QLIKE en el GRU, la cual superó a la reportada por los modelos GARCH y EGARCH ($\sigma \approx 0.06$ y $0.10$, respectivamente). Una hipótesis plausible para explicar este fenómeno radica en la naturaleza de los algoritmos de optimización. Mientras que el GARCH utiliza una estimación por Máxima Verosimilitud que converge a un único óptimo, el GRU se entrena mediante descenso de gradiente estocástico sobre una superficie de pérdida no convexa. Esto, sumado a la inicialización aleatoria de los pesos en cada partición (folds), indujo una mayor varianza en los resultados. Adicionalmente, la alta flexibilidad de la red neuronal provocó que su desempeño fuera más volátil ante los distintos regímenes de volatilidad presentes en cada fold de validación.

En la tabla de mejores ensayos, se seleccionó la primera configuración al registrar el menor QLIKE medio (0.430 ± 0.109 frente a 0.438 ± 0.109), manteniendo la misma estructura compacta de 7 días de ventana, 128 unidades ocultas, una sola capa y regularización $\ell_2$ de $7.7 \times 10^{-4}$ para contener el sobreajuste y favorecer la generalización.

In [17]:
best_model = GRUModel(
    look_back=7, 
    hidden_size=128,
    num_layers=1, 
    dropout=0.299, 
    learning_rate=0.0065,
    weight_decay=6.5e-04, 
    max_epochs=200, 
    patience=20, 
    retrain_freq=1, 
    seed=42
)

start_time = time.time()
best_model.fit(y_train, rs_train)
end_time = time.time()

elapsed_time = end_time - start_time

El análisis de la curva de aprendizaje descarta la presencia de sobreajuste. En un escenario de sobreajuste clásico, la pérdida de entrenamiento continuaría disminuyendo mientras la de validación empezaría a incrementarse, abriendo una brecha progresiva. En este caso, la pérdida de validación se mantiene baja y por debajo de la de entrenamiento a lo largo del recorrido. Esta brecha negativa se explica por el sesgo de selección temporal del conjunto de datos: la partición cronológica destina el periodo de mayor dificultad predictiva (los shocks extremos de 2020) al entrenamiento, mientras que la validación (2021-2022) corresponde a un régimen de volatilidad más moderado y, por tanto, con un QLIKE intrínsecamente más bajo.

In [7]:
history_csv_path = gru_exec_dir / "gru_training_history.csv"
history_df = pd.read_csv(history_csv_path)

if "epoch" in history_df.columns:
    history_df = history_df.drop(columns=["epoch"])

history_from_csv = {
    col: history_df[col].dropna().tolist()
    for col in history_df.columns
}

tt.plot_learning_curve(
    history=history_from_csv,
    title="Curva de Aprendizaje",
    ylab = "Quasi-Likelihood (QLIKE)",
    xlab_pad=14, ylab_pad=14,
    line_width=2.0,
    title_size=14, label_size=14,
    tick_size=12, legend_size=14,
    ylim=(0, 6.5), xlim=(0, 24), y_step=0.5, x_step=2,
    legend_bbox=(0.95, 0.95)
)

El modelo alcanzó su punto de mínimo error en la época 3, punto en el cual se seleccionaron los mejores pesos y tras el cual el entrenamiento se extendió hasta la época 23. La curva de aprendizaje evidenció que, tras la caída inicial de las primeras épocas y una ligera oscilación intermedia, ambas pérdidas tendieron a estabilizarse y converger aproximadamente a partir de las épocas 15 y 16. Esta estabilización, sumada a la baja pérdida en validación, indica que la configuración seleccionada (una capa y 128 unidades ocultas) logró ajustarse de forma estable sin caer en sobreajuste, capturando el nivel general de la varianza condicional pero sin mejoras adicionales relevantes en los picos extremos tras el mínimo temprano.

Para evaluar el ajuste dentro de la muestra de entrenamiento, a diferencia de los modelos paramétricos, no es posible extraer los residuos directamente del objeto del modelo. En su lugar, se realizó un pase hacia adelante (*forward pass*) utilizando la totalidad de la serie de entrenamiento. Dado que la red neuronal fue entrenada con retornos estandarizados, la serie `y_train` se transformó previamente mediante el escalador (`scaler`) ajustado durante el `fit` y se organizó con `TimeSeriesDataset` en ventanas de rezagos según la longitud de contexto (`look_back = 7` días). Estas ventanas se evaluaron en modo `eval()` y bajo `torch.no_grad()`, acumulando las salidas de la red (`best_model.net`) y alineando las series de referencia (`y_train` y `rs_train`) a partir del índice `look_back` para exportar tanto el retorno observado (`y_true`) como la varianza de Rogers-Satchell (`rs_true`) junto con la varianza predicha (`y_pred`) en `gru_train_predictions.csv`.

In [46]:
best_model.net.eval()
look_back = best_model.context_length

y_train_scaled = best_model.scaler.transform(y_train.reshape(-1, 1)).flatten()

train_ds = TimeSeriesDataset(y_train_scaled, rs_train, look_back)
train_loader = DataLoader(train_ds, batch_size=len(train_ds), shuffle=False)

preds_list = []
with torch.no_grad():
    for x_batch, _ in train_loader:
        x_batch = x_batch.to(best_model.device)
        out = best_model.net(x_batch).cpu().numpy().flatten()
        preds_list.append(out)

preds_train_var = np.concatenate(preds_list)

y_train_aligned = y_train[look_back:]
rs_train_aligned = rs_train[look_back:]

df_train_preds = pd.DataFrame({
    "y_true": y_train_aligned,
    "rs_true": rs_train_aligned,
    "y_pred": preds_train_var,
})

train_output_path = gru_exec_dir / "gru_train_predictions.csv"
df_train_preds.to_csv(train_output_path, index=False)

Para evaluar la estructura de los residuos tanto en el conjunto de entrenamiento (In-Sample) como en el de prueba (Out-of-Sample), a partir de las varianzas predichas y los retornos reales, se calcularon los residuos estandarizados mediante la siguiente expresión:

$$
z_t = \frac{y_t}{\sqrt{\hat{\sigma}_t^2}}
$$

El análisis de los residuos sobre el conjunto de entrenamiento reveló un ajuste estructural adecuado. Las pruebas de Ljung-Box (rezagos 10, 20 y 30), ARCH-LM (rezagos 10, 20 y 30) y BDS (dimensión 2) no rechazaron sus hipótesis nulas ($p > .05$, tanto en valores crudos como ajustados por Bonferroni donde aplica). Lo anterior indica que el modelo logró absorber la autocorrelación serial, el agrupamiento de volatilidad condicional y la dependencia no lineal evaluada en los residuos estandarizados dentro de la muestra.

In [9]:
train_csv_path = gru_exec_dir / "gru_train_predictions.csv"
df_train_check = pd.read_csv(train_csv_path)

y_true_chk = df_train_check["y_true"].to_numpy(dtype=np.float64)
y_pred_chk = df_train_check["y_pred"].to_numpy(dtype=np.float64)

valid_mask = np.isfinite(y_true_chk) & np.isfinite(y_pred_chk) & (y_pred_chk > 0.0)
std_residuals_train = y_true_chk[valid_mask] / np.sqrt(y_pred_chk[valid_mask])
std_residuals_train = std_residuals_train[np.isfinite(std_residuals_train)]

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals_train,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística",
        "Estadístico",
        "p-valor",
        "p-adj",
        "Conclusión (α = 0.05)",
    ),
)

df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--"
)


def _format_p(x):
    if pd.isna(x):
        return "--"
    if x < 0.001:
        return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted


df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),5.9963,.816,1.000,Sin Autocorrelación
Ljung-Box (Lag 20),27.2894,.127,.382,Sin Autocorrelación
Ljung-Box (Lag 30),37.5025,.163,.489,Sin Autocorrelación
ARCH-LM (Lag 10),8.5300,.577,1.000,Sin Efectos ARCH
ARCH-LM (Lag 20),20.9052,.403,1.000,Sin Efectos ARCH
ARCH-LM (Lag 30),31.9712,.369,1.000,Sin Efectos ARCH
Jarque-Bera,1374.2889,< .001,--,Residuos No Normales
Lilliefors,0.0582,< .001,--,Residuos No Normales
BDS (Dim 2),0.6911,.489,--,Sin Estructura No Lineal


Las pruebas de Jarque-Bera y Lilliefors rechazaron la hipótesis nula de normalidad ($p < .001$), evidenciando la presencia de colas pesadas y asimetría en los residuos estandarizados, un rasgo habitual en los activos financieros. En el análisis gráfico, el ajuste de una distribución $t$ de Student ($\nu = 4.3913$) acompaña el cuerpo central de la distribución y sigue de cerca la recta diagonal en el gráfico Q-Q, aunque se observan ligeras desviaciones en los cuantiles extremos. A pesar de estos apartamientos puntuales, las pruebas formales de bondad de ajuste para la distribución $t$ de Student (Kolmogorov-Smirnov: $p = .3792$; Cramér-von Mises: $p = .3244$) no rechazaron la hipótesis nula ($\alpha = 0.05$).

In [12]:
_, _ = tst.plot_distribution_fit(
    pd.Series(std_residuals_train),
    dist="t",
    titles=("Distribución de Residuos", "Gráfico Q-Q (t-Student)"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 0.7),
    legend_loc="upper left"
)

La prueba de razón de verosimilitud (LRT) rechazó la hipótesis nula de igualdad de ajuste ($p < .001$). Los criterios de información evidenciaron una reducción de $\Delta\text{AIC} = 231.43$ y $\Delta\text{BIC} = 225.75$ a favor de la distribución $t$ de Student frente a la distribución normal. Esto indica que la inclusión del parámetro de grados de libertad ($\nu = 4.3913$) captura mejor la distribución empírica de colas pesadas de los residuos estandarizados en la muestra de entrenamiento en comparación con la distribución gaussiana.

Es importante precisar que, a diferencia de los modelos GARCH y EGARCH, el GRU no impone una distribución paramétrica específica durante su estimación, dado que la pérdida QLIKE se optimiza directamente sobre la varianza. Por tanto, este contraste no valida un supuesto interno del modelo, sino que verifica empíricamente que la distribución $t$ describe mejor la densidad de colas pesadas de los residuos que la distribución normal.

In [13]:
_= tst.distribution_fit_table(
    pd.Series(std_residuals_train),
    column_titles=(
        "Distribución", 
        "Log-Likelihood", 
        "AIC", 
        "BIC", 
        "Estadístico LRT", 
        "p-valor LRT"
    )
)

Distribución,Log-Likelihood,AIC,BIC,Estadístico LRT,p-valor LRT
Normal,"-3,291.36","6,586.71","6,598.07",-,-
t-Student,"-3,174.64","6,355.28","6,372.32",233.43,< .001


La estructura del GRU logró seguir la trayectoria de la volatilidad realizada y reproducir el surgimiento de eventos de alta volatilidad, como la crisis del COVID-19 en 2020 y los repuntes de 2022. En el detalle de la crisis de 2020, se observó que la predicción capturó el repunte de marzo (cerca de 0.22) y parte del movimiento de abril (alrededor de 0.15), aunque presentó subestimaciones en otros picos puntuales de volatilidad realizada que alcanzaron valores cercanos a 0.25. A pesar de estas diferencias en la magnitud máxima de los extremos, la arquitectura permitió capturar el agrupamiento y la dinámica temporal de la volatilidad a lo largo de todo el periodo histórico.

In [17]:
look_back = 7
dates_aligned = df_train.index[look_back:]
rs_volatilidad_aligned = np.sqrt(df_train_check["rs_true"].to_numpy(dtype=np.float64))
pred_volatilidad = np.sqrt(df_train_check["y_pred"].to_numpy(dtype=np.float64))

mask_crisis = (dates_aligned >= "2020-01-01") & (dates_aligned <= "2020-06-30")

ts.plot_volatility_fit(
    dates=dates_aligned,
    rs_variance=rs_volatilidad_aligned,
    pred_variance=pred_volatilidad,
    title="Ajuste dentro la Muestra de Entrenamiento (GRU)",
    ylab="Volatilidad",
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$",
    label_pred=r"Volatilidad GRU $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.35),
    legend_bbox=(0.01, 0.98),
    zoom_slice=mask_crisis,
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Crisis COVID-19",
    zoom_bbox=(0.18, 0.35, 0.40, 0.40),
    title_size=14,
    label_size=14,
    tick_size=12,
    legend_size=14,
)

Este nivel de ajuste resultó esperado al tratarse de la muestra de entrenamiento. Al ser evaluada con los mismos datos utilizados para ajustar sus parámetros, la red neuronal desplegó su capacidad para mapear la secuencia histórica. Por lo tanto, para determinar si la arquitectura poseyó verdadera capacidad de generalización ante nuevos regímenes de mercado, se procedió a la evaluación sobre el conjunto de prueba (Out-of-Sample).

## Evaluación

La evaluación del rendimiento de un modelo de volatilidad requirió contrastar su capacidad de ajuste sobre los datos observados con su capacidad predictiva sobre datos no vistos. Mientras que el conjunto de entrenamiento (In-Sample) permitió verificar la convergencia del algoritmo a la dinámica histórica, la evaluación Out-of-Sample (conjunto de prueba) determinó el desempeño predictivo del modelo ante nuevos regímenes de mercado.

Para diagnosticar la presencia de sobreajuste (*overfitting*) o falta de generalización, se aplicaron los criterios pre-registrados en el protocolo experimental. Estos criterios se basaron en la diferencia entre las métricas de prueba y entrenamiento, definida como:

$$
\Delta_{abs} = M_{test} - M_{train}
$$

Se estableció un umbral de referencia absoluto de $|\Delta_{abs}| = 0.05$ para las métricas de evaluación. Un valor de $|\Delta_{abs}|$ superior a este margen señaló una brecha relevante entre ambos conjuntos.

En el contexto de la varianza condicional, es importante considerar la asimetría de la pérdida QLIKE. Dado que QLIKE penaliza con mayor severidad la subestimación de la varianza realizada frente a la sobreestimación, una diferencia negativa $\left(\Delta_{abs} < 0\right)$ no implica automáticamente una mejor generalización, ya que la dispersión o magnitud de los choques en el periodo de prueba puede diferir de la muestra de entrenamiento. Por ello, el análisis de $\Delta_{abs}$ se complementó con la inspección de las trayectorias predichas y los diagnósticos de residuos estandarizados.

Las predicciones fuera de muestra se obtuvieron mediante el método `forecast` del wrapper del modelo, bajo un esquema *walk-forward* con reentrenamiento periódico sin fuga de información, generando las varianzas pronosticadas en la escala de varianza objetivo para su comparación directa con la varianza de Rogers-Satchell.

In [ ]:
start_time = time.time()
preds_test = best_model.forecast(y_seq=y_test, rs_seq=rs_test) 
preds_time = time.time() - start_time

df_preds = pd.DataFrame({
    "date": df_test.index, 
    "rs_true": rs_test, 
    "y_pred": preds_test
})

output_path = RESULTS_PATH / "gru_test_predictions.csv"
df_preds.to_csv(output_path, index=False)

La generación de pronósticos fuera de muestra mediante el esquema de ventana expansiva y reentrenamiento periódico tuvo una duración total de 113.16 minutos para las 953 observaciones del conjunto de prueba, lo que equivale a un tiempo promedio de 0.12 minutos (unos 7.12 segundos) por observación. Este tiempo es sustancialmente superior al de los modelos paramétricos, cuyo cómputo toma milisegundos, reflejando el costo computacional de actualizar y reajustar los pesos de la red neuronal a lo largo de la ventana de evaluación.

El diagnóstico de los residuos estandarizados fuera de muestra mostró un patrón opuesto al del diagnóstico previo: las pruebas ARCH-LM no rechazaron la hipótesis nula en ninguno de los rezagos evaluados, y la prueba BDS en dimensión 2 tampoco rechazó la ausencia de dependencia no lineal. Por el contrario, las pruebas de Ljung-Box mostraron autocorrelación residual significativa en los rezagos 20 y 30, mientras que en el rezago 10 el valor ajustado no alcanzó significancia tras el ajuste de Bonferroni.

In [18]:
import numpy as np
import pandas as pd

preds_path = RESULTS_PATH / "exec_2" / "gru_test_predictions.csv"
df_preds_test = pd.read_csv(preds_path)

y_test_arr = np.asarray(y_test, dtype=np.float64).ravel()
preds_test_var = df_preds_test["y_pred"].to_numpy(dtype=np.float64).ravel()

valid_mask = (
    np.isfinite(y_test_arr)
    & np.isfinite(preds_test_var)
    & (preds_test_var > 0.0)
)
std_residuals_oos = y_test_arr[valid_mask] / np.sqrt(preds_test_var[valid_mask])
std_residuals_oos = std_residuals_oos[np.isfinite(std_residuals_oos)]

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals_oos,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística",
        "Estadístico",
        "p-valor",
        "p-adj",
        "Conclusión (α = 0.05)",
    ),
)

df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--"
)


def _format_p(x):
    if pd.isna(x):
        return "--"
    if x < 0.001:
        return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted


df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),20.4713,.025,.075,Sin Autocorrelación
Ljung-Box (Lag 20),41.3226,.003,.010,Autocorrelación Significativa
Ljung-Box (Lag 30),49.6037,.014,.041,Autocorrelación Significativa
ARCH-LM (Lag 10),10.1281,.429,1.000,Sin Efectos ARCH
ARCH-LM (Lag 20),19.4816,.491,1.000,Sin Efectos ARCH
ARCH-LM (Lag 30),38.5037,.137,.412,Sin Efectos ARCH
Jarque-Bera,13202.9962,< .001,--,Residuos No Normales
Lilliefors,0.0555,< .001,--,Residuos No Normales
BDS (Dim 2),-0.8796,.379,--,Sin Estructura No Lineal


Estos resultados indican que, aunque el modelo logró mitigar la heterocedasticidad condicional residual y no dejó dependencia no lineal en la dimensión evaluada, persistió cierta autocorrelación serial en horizontes de rezago intermedios y mayores en datos no vistos, lo que sugiere que la dinámica temporal de mediano plazo aún presenta margen de ajuste.

Las pruebas de normalidad rechazaron la hipótesis nula gaussiana $\left(p < .001\right)$, indicando que la distribución normal no capturó las colas pesadas de los residuos estandarizados fuera de muestra. La inspección visual mostró que el histograma de densidad se ajustó de cerca a una distribución $t$ de Student $\left(\nu = 5.1612\right)$ y que el gráfico Q-Q acompañó la diagonal en su tramo central, con una desviación en el cuantil inferior extremo donde el valor observado cayó por debajo de la recta teórica, reflejando una subestimación de la volatilidad ante un retorno negativo extremo. Las pruebas de bondad de ajuste (Kolmogorov-Smirnov: $p = .9081$; Cramér-von Mises: $p = .7897$) apoyaron la hipótesis de pertenencia a la distribución $t$ de Student.

In [20]:
_, _ = tst.plot_distribution_fit(
    pd.Series(std_residuals_oos),
    dist="t",
    titles=("Distribución de Residuos", "Gráfico Q-Q (t-Student)"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 0.7),
    legend_loc="upper left"
)

La prueba de razón de verosimilitud (LRT) rechazó la hipótesis nula de igualdad de ajuste ($p < .001$). Los criterios de información evidenciaron una reducción de $\Delta\text{AIC} = 173.35$ y $\Delta\text{BIC} = 168.50$ a favor de la distribución $t$ de Student frente a la distribución normal. Esto indicó que la inclusión del parámetro de grados de libertad $\left(\nu = 5.1612\right)$ capturó mejor la distribución empírica de colas pesadas de los residuos estandarizados en el conjunto de prueba en comparación con la distribución gaussiana.

In [27]:
_= tst.distribution_fit_table(
    pd.Series(std_residuals_oos),
    column_titles=(
        "Distribución", 
        "Log-Likelihood", 
        "AIC", 
        "BIC", 
        "Estadístico LRT", 
        "p-valor LRT"
    )
)

Distribución,Log-Likelihood,AIC,BIC,Estadístico LRT,p-valor LRT
Normal,"-1,474.97","2,953.94","2,963.66",-,-
t-Student,"-1,387.29","2,780.59","2,795.16",175.35,< .001


El modelo GRU no cumplió con el criterio de tolerancia pre-fijado para la métrica principal, ya que la pérdida QLIKE en prueba superó a la de entrenamiento con una diferencia positiva que excedió el umbral de 0.05, lo que evidenció una caída en el desempeño predictivo fuera de muestra frente al ajuste histórico; aunque este salto refleja pérdida de generalización, también responde a la mayor dificultad para calibrar la volatilidad ante cambios de régimen y picos extremos. Las métricas probabilísticas (CRPS y Pinball) y los errores puntuales (MSE, RMSE, MAE) se mantuvieron prácticamente estables entre ambos conjuntos, indicando que el modelo conservó el nivel general de la serie. En contraste, las métricas ajustadas por heterocedasticidad registraron aumentos marcados en prueba, destacando el salto de HMSE y el incremento de HMAE, consistentes con la mayor sensibilidad de estas medidas ante desvíos en episodios de alta volatilidad.

In [28]:
df_train_preds = pd.read_csv(gru_exec_dir / "gru_train_predictions.csv")
df_test_preds = pd.read_csv(gru_exec_dir / "gru_test_predictions.csv")

ts.display_train_test_comparison(
    rs_train=df_train_preds["rs_true"].to_numpy(),
    preds_train=df_train_preds["y_pred"].to_numpy(),
    rs_test=df_test_preds["rs_true"].to_numpy(),
    preds_test=df_test_preds["y_pred"].to_numpy(),
    column_titles=(
        "Métrica",
        "Entrenamiento",
        "Prueba",
        "Diferencia",
    ),
)

Métrica,Entrenamiento,Prueba,Diferencia
QLIKE,0.4097,0.5109,+0.1012
CRPS,0.0127,0.0124,-0.0003
Pinball (τ=0.05),0.0027,0.0027,+0.0000
Pinball (τ=0.95),0.0014,0.0017,+0.0003
MSE,0.0000,0.0000,-0.0000
RMSE,0.0021,0.0020,-0.0001
MAE,0.0004,0.0005,+0.0001
HMSE,1.4770,6.4425,+4.9655
HMAE,0.6883,0.8131,+0.1248


El incremento del HMSE en prueba coincidió con el comportamiento observado en la trayectoria fuera de muestra, donde la predicción del GRU acompañó la dinámica general pero presentó oscilaciones frecuentes y desajustes en los picos de volatilidad. En el detalle del episodio de 2026, la volatilidad realizada alcanzó un máximo cercano a 0.25 en marzo, mientras que la estimación del GRU quedó notoriamente por debajo (alrededor de 0.11), alternando posteriormente entre leves sobreestimaciones y subestimaciones en los meses siguientes. Dado que QLIKE penaliza con mayor severidad la subestimación de la varianza realizada frente a la sobreestimación, ese rezago o contención en la cima del shock impactó de forma directa el desempeño fuera de muestra.

A diferencia de los modelos parámetricos, la arquitectura GRU mostró una respuesta más reactiva ante las variaciones recientes de los datos de entrada, lo que generó fluctuaciones pronunciadas en regímenes turbulentos como el tramo de 2026. Sin embargo, esa reactividad no bastó para calibrar la magnitud máxima del shock principal ni evitar oscilaciones posteriores, confirmando que la flexibilidad recurrente por sí sola no resolvió la complejidad de los picos extremos en datos no vistos.

In [24]:
df_preds_test = df_preds_test.copy()
df_preds_test["date"] = pd.to_datetime(df_preds_test["date"])

rs_test_var = df_preds_test["rs_true"].to_numpy(dtype=np.float64)
pred_test_var = df_preds_test["y_pred"].to_numpy(dtype=np.float64)

rs_test_vol = np.sqrt(np.maximum(rs_test_var, 0.0))
pred_test_vol = np.sqrt(np.maximum(pred_test_var, 0.0))

mask_crisis = (
    (df_preds_test["date"] >= "2026-02-01")
    & (df_preds_test["date"] <= "2026-06-30")
).to_numpy()

ts.plot_volatility_fit(
    dates=df_preds_test["date"],
    rs_variance=rs_test_vol,
    pred_variance=pred_test_vol,
    title="Ajuste Fuera de la Muestra (GRU)",
    ylab="Volatilidad",
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$",
    label_pred=r"Volatilidad GRU $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.35),
    legend_bbox=(0.01, 0.98),
    zoom_slice=mask_crisis,
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Pico de Volatilidad (2026)",
    zoom_bbox=(0.25, 0.35, 0.40, 0.40),
    title_size=14,
    label_size=14,
    tick_size=12,
    legend_size=14,
)

## Conclusiones

La implementación y evaluación del modelo GRU permitieron contrastar el desempeño de esta arquitectura recurrente para el pronóstico de la volatilidad diaria del crudo Brent. En entrenamiento, el modelo alcanzó convergencia estable y presentó residuos estandarizados sin autocorrelación significativa ni efectos ARCH en los rezagos evaluados, con una estructura de colas pesadas mejor descrita por la distribución $t$ de Student que por la normal. Sin embargo, en la evaluación fuera de muestra, la pérdida QLIKE aumentó por encima del umbral de tolerancia establecido respecto a la muestra de entrenamiento, y los diagnósticos mostraron autocorrelación residual significativa en los rezagos de mayor horizonte (20 y 30), sin presencia de efectos ARCH ni dependencia no lineal en dimensión 2.

La selección de hiperparámetros con Optuna favoreció una ventana de contexto de 7 días, 128 unidades ocultas y una capa recurrente, junto con regularización moderada. Esta configuración permitió capturar el nivel general de la volatilidad y responder a los cambios de régimen sin requerir mayor profundidad, pero mostró limitaciones para calibrar la magnitud de los eventos extremos en el periodo de prueba, como se observó en el pico de marzo de 2026 donde la estimación quedó por debajo del máximo realizado.

La asimetría de la función de pérdida QLIKE, que castiga con mayor severidad la subestimación de la varianza realizada frente a la sobreestimación, amplificó el impacto de las subestimaciones en picos abruptos y se reflejó en el incremento de las métricas ajustadas por heterocedasticidad (HMSE y HMAE) fuera de muestra. El esquema *walk-forward* con reentrenamiento periódico implicó un tiempo total de 113.16 minutos para las 953 observaciones de prueba, evidenciando un costo computacional considerablemente mayor al de los modelos paramétricos. En conjunto, los resultados indican que la arquitectura GRU captura la dinámica general y la reactividad de corto plazo de la volatilidad, pero requiere mayor precisión en la calibración de extremos y menor costo de cómputo para superar de forma consistente a los enfoques de referencia.